In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)  # Makes results reproducible - same seed = same data every time
N = 600

# ── Identifiers ────────────────────────────────────────────────────────────────
customer_ids = [f"CUST-{str(i).zfill(5)}" for i in range(1, N+1)]

first_names = ["Thabo", "Sipho", "Nomsa", "Lerato", "Pieter", "Zanele", "Johan", "Ayanda",
               "Fatima", "Ruan", "Nokuthula", "Dlamini", "Heinrich", "Precious", "Andre",
               "Nompumelelo", "Dawie", "Thandeka", "Christiaan", "Bongiwe", "Yusuf", "Amahle",
               "Kobus", "Sindisiwe", "Petrus", "Lungelo", "Morne", "Ntombifuthi", "Francois",
               "Nozipho", "Riaan", "Hlengiwe", "Sarel", "Busisiwe", "Gerrit", "Khanyisile"]

last_names = ["Dlamini", "Nkosi", "Van der Merwe", "Zulu", "Botha", "Ndlovu", "Pretorius",
              "Mthembu", "Pieterse", "Khumalo", "Du Plessis", "Mhlongo", "Kruger", "Ntuli",
              "Venter", "Cele", "Steyn", "Mabaso", "Olivier", "Shabalala", "Adams", "Mahlangu",
              "Bester", "Nxumalo", "Coetzee", "Sibiya", "Joubert", "Mnguni", "Fourie", "Khoza"]

first = np.random.choice(first_names, N)
last  = np.random.choice(last_names, N)

# ── Demographics ───────────────────────────────────────────────────────────────
age    = np.random.normal(38, 12, N).clip(18, 75).astype(int)
gender = np.random.choice(["Male", "Female"], N, p=[0.48, 0.52])

education = np.random.choice(
    ["Matric", "Diploma", "Undergraduate", "Postgraduate"],
    N, p=[0.35, 0.25, 0.28, 0.12]
)

# Income: base driven by education, boosted by age (experience proxy)
edu_income_base = {"Matric": 8000, "Diploma": 15000, "Undergraduate": 25000, "Postgraduate": 45000}
base_income    = np.array([edu_income_base[e] for e in education])
age_boost      = (age - 18) * 300
monthly_income = (base_income + age_boost + np.random.normal(0, 5000, N)).clip(3500, 150000).astype(int)

income_band = pd.cut(
    monthly_income,
    bins=[0, 10000, 25000, 50000, 100000, np.inf],
    labels=["Low", "Lower-Middle", "Middle", "Upper-Middle", "High"]
).astype(str)

employment_status = np.where(
    age < 23,
    np.random.choice(["Student", "Part-time", "Employed"], N, p=[0.6, 0.2, 0.2]),
    np.where(
        age > 60,
        np.random.choice(["Retired", "Self-employed", "Employed"], N, p=[0.5, 0.2, 0.3]),
        np.random.choice(["Employed", "Self-employed", "Unemployed", "Contract"], N, p=[0.60, 0.18, 0.10, 0.12])
    )
)

province = np.random.choice(
    ["Gauteng", "Western Cape", "KwaZulu-Natal", "Eastern Cape",
     "Limpopo", "Mpumalanga", "North West", "Free State", "Northern Cape"],
    N, p=[0.26, 0.17, 0.20, 0.12, 0.08, 0.07, 0.05, 0.04, 0.01]
)

marital_status = np.where(
    age < 25,
    np.random.choice(["Single", "Married", "Divorced"], N, p=[0.85, 0.12, 0.03]),
    np.where(
        age < 40,
        np.random.choice(["Single", "Married", "Divorced", "Widowed"], N, p=[0.40, 0.48, 0.10, 0.02]),
        np.random.choice(["Single", "Married", "Divorced", "Widowed"], N, p=[0.15, 0.58, 0.20, 0.07])
    )
)

number_of_dependents = np.where(
    marital_status == "Single",
    np.random.choice([0, 1, 2], N, p=[0.65, 0.25, 0.10]),
    np.where(
        marital_status == "Married",
        np.random.choice([0, 1, 2, 3, 4], N, p=[0.15, 0.25, 0.35, 0.18, 0.07]),
        np.random.choice([0, 1, 2, 3], N, p=[0.30, 0.35, 0.25, 0.10])
    )
)

# ── Account ─────────────────────────────────────────────────────────────────────
account_tenure_years = ((age - 18) * 0.5 + np.random.normal(0, 3, N)).clip(0, 40).astype(int)

# ── Risk ────────────────────────────────────────────────────────────────────────
employment_score_boost = np.where(employment_status == "Employed",      50,
                         np.where(employment_status == "Self-employed",  20,
                         np.where(employment_status == "Retired",        30, -30)))

credit_score = (500 + (monthly_income / 150000) * 250 + employment_score_boost +
                np.random.normal(0, 40, N)).clip(300, 850).astype(int)

loan_utilisation_rate = ((900 - credit_score) / 600 + np.random.normal(0, 0.1, N)).clip(0.0, 1.0).round(2)

days_since_last_default = np.where(
    credit_score > 700,
    np.random.choice([999, np.random.randint(500, 1500)], N, p=[0.75, 0.25]),
    np.where(credit_score > 550, np.random.randint(30, 730, N), np.random.randint(0, 365, N))
).astype(int)
# NOTE: 999 is used as a sentinel value meaning "never defaulted / no record"

late_payment_count = np.where(
    credit_score > 700, np.random.choice([0, 1], N, p=[0.85, 0.15]),
    np.where(credit_score > 550, np.random.choice([0,1,2,3], N, p=[0.50,0.25,0.15,0.10]),
    np.random.choice([0,1,2,3,4,5], N, p=[0.20,0.20,0.25,0.20,0.10,0.05]))
)

# ── Behavioural ─────────────────────────────────────────────────────────────────
monthly_transactions    = (monthly_income / 3000 + np.random.normal(0, 5, N)).clip(1, 80).astype(int)
avg_transaction_value   = (monthly_income / (monthly_transactions + 1) * np.random.uniform(0.3, 0.7, N)).clip(50, 20000).round(2)
monthly_expenses        = (monthly_income * np.random.uniform(0.5, 0.95, N)).clip(2000, 130000).round(2)
digital_login_frequency = ((40 - age) * 0.3 + (monthly_income / 10000) + np.random.normal(10, 5, N)).clip(0, 60).astype(int)
branch_visit_count      = ((age / 10) + np.random.normal(2, 2, N)).clip(0, 15).astype(int)
preferred_channel       = np.where(digital_login_frequency > 20, "Mobile",
                          np.where(digital_login_frequency > 10, "Online", "Branch"))

# ── Products ────────────────────────────────────────────────────────────────────
has_savings_account  = np.ones(N, dtype=int)  # All customers have a base account
has_credit_card      = ((credit_score > 580) & (monthly_income > 8000)  & (np.random.random(N) < 0.65)).astype(int)
has_home_loan        = ((age > 25) & (monthly_income > 20000) & (credit_score > 620) &
                         np.isin(marital_status, ["Married", "Divorced"]) &
                        (np.random.random(N) < 0.45)).astype(int)
has_personal_loan    = ((credit_score > 550) & (monthly_income > 10000) & (np.random.random(N) < 0.40)).astype(int)
has_vehicle_finance  = ((age > 22) & (monthly_income > 15000) & (credit_score > 580) & (np.random.random(N) < 0.38)).astype(int)
has_investment_account = ((monthly_income > 30000) & (credit_score > 650) & (age > 28) & (np.random.random(N) < 0.35)).astype(int)
active_products_count  = (has_savings_account + has_credit_card + has_home_loan +
                           has_personal_loan + has_vehicle_finance + has_investment_account)

# ── Churn Risk Label ─────────────────────────────────────────────────────────────
# Composite score: high churn = low engagement + few products + high risk indicators
churn_score = (
    - (digital_login_frequency  * 0.8)
    - (active_products_count    * 5.0)
    - (account_tenure_years     * 1.5)
    - (credit_score             * 0.02)
    + (loan_utilisation_rate    * 15.0)
    + (late_payment_count       * 4.0)
    + np.random.normal(0, 8, N)
)
churn_percentiles = np.percentile(churn_score, [45, 75])
churn_risk_label  = np.where(churn_score < churn_percentiles[0], "Low",
                    np.where(churn_score < churn_percentiles[1], "Medium", "High"))

# ── Assemble & Save ──────────────────────────────────────────────────────────────
df = pd.DataFrame({
    "customer_id": customer_ids, "first_name": first, "last_name": last,
    "age": age, "gender": gender, "marital_status": marital_status,
    "number_of_dependents": number_of_dependents, "education_level": education,
    "province": province, "employment_status": employment_status,
    "monthly_income": monthly_income, "income_band": income_band,
    "monthly_expenses": monthly_expenses, "account_tenure_years": account_tenure_years,
    "monthly_transactions": monthly_transactions, "avg_transaction_value": avg_transaction_value,
    "digital_login_frequency": digital_login_frequency, "branch_visit_count": branch_visit_count,
    "preferred_channel": preferred_channel, "late_payment_count": late_payment_count,
    "has_savings_account": has_savings_account, "has_credit_card": has_credit_card,
    "has_home_loan": has_home_loan, "has_personal_loan": has_personal_loan,
    "has_vehicle_finance": has_vehicle_finance, "has_investment_account": has_investment_account,
    "active_products_count": active_products_count, "credit_score": credit_score,
    "loan_utilisation_rate": loan_utilisation_rate,
    "days_since_last_default": days_since_last_default,
    "churn_risk_label": churn_risk_label
})

df.to_csv("../data/bank_customers.csv", index=False)
print(f"✓ Dataset saved: {df.shape[0]} customers, {df.shape[1]} features")
print(df.head())

✓ Dataset saved: 600 customers, 31 features
  customer_id  first_name  last_name  age  gender marital_status  \
0  CUST-00001    Francois    Mthembu   19    Male         Single   
1  CUST-00002       Andre  Pretorius   71  Female       Divorced   
2  CUST-00003      Ayanda     Sibiya   57    Male         Single   
3  CUST-00004       Yusuf     Sibiya   37    Male        Married   
4  CUST-00005  Christiaan      Nkosi   34  Female        Married   

   number_of_dependents education_level       province employment_status  ...  \
0                     1          Matric   Eastern Cape          Employed  ...   
1                     1         Diploma   Eastern Cape           Retired  ...   
2                     0          Matric     Mpumalanga        Unemployed  ...   
3                     4   Undergraduate        Gauteng          Employed  ...   
4                     2         Diploma  KwaZulu-Natal     Self-employed  ...   

   has_credit_card has_home_loan  has_personal_loan  has_veh